# Task 3 — Linear SVM from scratch (archived working notebook)

A hinge-loss linear SVM trained by mini-batch gradient descent on its own
word 1–2 gram TF-IDF, with the `C` and `class_weight` sweep the member brief
asks for.

**It is not the graded deliverable**, and it is kept for the record rather
than for its score. Two things are worth reading off it:

- Without `class_weight="balanced"` every value of `C` scores 0.3847 — exactly
  the always-predict-1 baseline. On a 62.5/37.5 split the unbalanced hinge
  loss simply collapses onto the majority class, so `C` cannot matter until
  the classes are reweighted. Balancing lifts it to 0.6586.
- The same hinge loss, reached through `sgd_fit` in Section 3.3 of the final
  notebook, scores 0.7446 on the provided features and about 0.85 once the
  hybrid TF-IDF and stylometry features replace them.

It writes only to `submissions/experiments/` so re-running it cannot overwrite
a graded file.


In [19]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

SEED = 42
VALIDATION_SIZE = 0.20

# Works when the notebook is opened either from the repository root or notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "train.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "train.csv").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find data/train.csv from the current folder.")

TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / "shared_validation_split.csv"
OUTPUT_PATH = PROJECT_ROOT / "submissions" / "experiments" / "SVM_Prediction.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

ID_COLUMN = "id"
LABEL_COLUMN = "label"
TEXT_COLUMN = "text"
FEATURE_COLUMNS = [c for c in train.columns if c not in {ID_COLUMN, LABEL_COLUMN}]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("ID column:", ID_COLUMN)
print("Label column:", LABEL_COLUMN)
print("Feature/input columns:", FEATURE_COLUMNS)
display(train.head())

Train shape: (20000, 3)
Test shape: (6999, 2)
ID column: id
Label column: label
Feature/input columns: ['text']


,id,text,label
0,71ec6000-1f20-4850-a8f7-140bc6ad640d,Instance-level video segmentation requires a s...,1
1,9a494ddf-43c6-4c54-8927-f2343054fef2,Samples of high-redshift galaxies are easy to ...,0
2,bc1dc7e4-d773-4c52-bcc1-ed08ba822f92,"Ashley Sibery, 39, persuaded Sital Sibery to t...",1
3,c9f43db6-21b9-4ca6-9830-ce06ff47b950,On-shell methods offer an alternative definiti...,0
4,3c0ef5fb-5311-4185-b773-ea48e7e5bd51,The ocean goes through two tide cycles in a da...,1


In [20]:
summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "missing_values": [int(train.isna().sum().sum()), int(test.isna().sum().sum())],
    "missing_text": [int(train[TEXT_COLUMN].isna().sum()), int(test[TEXT_COLUMN].isna().sum())],
})

display(summary)

class_distribution = (
    train[LABEL_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)
class_distribution["percentage"] = 100 * class_distribution["count"] / len(train)
display(class_distribution)

print("Raw input feature columns:", FEATURE_COLUMNS)
print("The actual TF-IDF feature count will be printed after vectorisation below.")

,dataset,rows,columns,missing_values,missing_text
0,train,20000,3,0,0
1,test,6999,2,0,0


,label,count,percentage
0,0,7496,37.48
1,1,12504,62.52


Raw input feature columns: ['text']
The actual TF-IDF feature count will be printed after vectorisation below.


In [22]:
def make_stratified_split(y, validation_size=0.20, seed=42):
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    split = np.full(len(y), "train", dtype=object)

    for label in np.unique(y):
        idx = np.flatnonzero(y == label).copy()
        rng.shuffle(idx)
        n_val = int(round(len(idx) * validation_size))
        split[idx[:n_val]] = "validation"

    return split

SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.exists():
    shared_split = pd.read_csv(SPLIT_PATH)
    print("Loaded existing team split:", SPLIT_PATH)
else:
    split_values = make_stratified_split(
        train[LABEL_COLUMN].to_numpy(),
        validation_size=VALIDATION_SIZE,
        seed=SEED,
    )
    shared_split = pd.DataFrame({
        "row_index": np.arange(len(train)),
        "id": train[ID_COLUMN].astype(str),
        "label": train[LABEL_COLUMN].astype(int),
        "split": split_values,
    })
    shared_split.to_csv(SPLIT_PATH, index=False)
    print("Created and saved team split:", SPLIT_PATH)

# Verify alignment using row_index if present, otherwise IDs.
if len(shared_split) != len(train):
    raise ValueError("Shared split row count does not match train.csv")

if "row_index" in shared_split.columns:
    if not np.array_equal(shared_split["row_index"].to_numpy(), np.arange(len(train))):
        raise ValueError("Shared split row_index does not align with train.csv")
elif "id" in shared_split.columns:
    if shared_split["id"].astype(str).tolist() != train[ID_COLUMN].astype(str).tolist():
        raise ValueError("Shared split IDs do not align with train.csv")

train_mask = shared_split["split"].eq("train").to_numpy()
validation_mask = shared_split["split"].eq("validation").to_numpy()

print("Training rows:", int(train_mask.sum()))
print("Validation rows:", int(validation_mask.sum()))
display(pd.crosstab(shared_split["split"], train[LABEL_COLUMN], margins=True))

Loaded existing team split: c:\Users\akshi\Documents\github\TheHomiesML\data\splits\shared_validation_split.csv
Training rows: 16000
Validation rows: 4000


label,0,1,All
split,,,
train,5997,10003,16000
validation,1499,2501,4000
All,7496,12504,20000


In [23]:
def macro_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []

    for label in (0, 1):
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))
        denominator = 2 * tp + fp + fn
        scores.append(0.0 if denominator == 0 else 2 * tp / denominator)

    return float(np.mean(scores))

In [24]:
def create_submission(test_ids, predictions, output_path):
    predictions = np.asarray(predictions)
    if len(test_ids) != len(predictions):
        raise ValueError("Prediction count does not match test IDs.")
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError("Predictions must contain only 0 and 1.")

    submission = pd.DataFrame({
        "id": pd.Series(test_ids).astype(str),
        "label": predictions.astype(np.int8),
    })
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)
    return submission

In [25]:
TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")

class ScratchTfidfVectorizer:
    def __init__(self, min_df=2, max_features=100_000, ngram_range=(1, 2)):
        self.min_df = min_df
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.vocabulary_ = {}
        self.idf_ = None

    def _terms(self, document):
        tokens = TOKEN_RE.findall(str(document).lower())
        terms = []
        if self.ngram_range[0] <= 1 <= self.ngram_range[1]:
            terms.extend(tokens)
        if self.ngram_range[0] <= 2 <= self.ngram_range[1]:
            terms.extend(a + " " + b for a, b in zip(tokens, tokens[1:]))
        return terms

    def fit(self, documents):
        documents = list(documents)
        df = Counter()
        for document in documents:
            df.update(set(self._terms(document)))

        eligible = [(term, count) for term, count in df.items() if count >= self.min_df]
        selected = sorted(eligible, key=lambda x: (-x[1], x[0]))[:self.max_features]
        self.vocabulary_ = {term: i for i, (term, _) in enumerate(selected)}
        dfs = np.asarray([count for _, count in selected], dtype=np.float64)
        self.idf_ = np.log((1.0 + len(documents)) / (1.0 + dfs)) + 1.0
        return self

    def transform(self, documents):
        if self.idf_ is None:
            raise RuntimeError("Fit the vectorizer before transform().")

        documents = list(documents)
        rows, cols, vals = [], [], []
        for row, document in enumerate(documents):
            counts = Counter(self._terms(document))
            for term, count in counts.items():
                col = self.vocabulary_.get(term)
                if col is not None:
                    rows.append(row)
                    cols.append(col)
                    vals.append(1.0 + np.log(count))

        X = sparse.csr_matrix(
            (np.asarray(vals) * self.idf_[cols], (rows, cols)),
            shape=(len(documents), len(self.vocabulary_)),
            dtype=np.float64,
        )
        norms = np.sqrt(X.multiply(X).sum(axis=1)).A1
        norms[norms == 0] = 1.0
        return sparse.diags(1.0 / norms).dot(X).tocsr()

    def fit_transform(self, documents):
        documents = list(documents)
        return self.fit(documents).transform(documents)

In [26]:
class ScratchLinearSVM:
    def __init__(self, C=1.0, epochs=20, batch_size=256,
                 learning_rate=0.5, decay=0.02,
                 class_weight=None, random_state=42):
        self.C = C
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.decay = decay
        self.class_weight = class_weight
        self.random_state = random_state
        self.weights_ = None
        self.bias_ = 0.0

    def fit(self, X, y, verbose=False):
        y = np.asarray(y, dtype=np.int8)
        signed_y = np.where(y == 1, 1.0, -1.0)
        sample_weight = np.ones(len(y), dtype=np.float64)

        if self.class_weight == "balanced":
            counts = np.bincount(y, minlength=2)
            sample_weight = np.asarray([
                len(y) / (2 * counts[label]) for label in y
            ])

        self.weights_ = np.zeros(X.shape[1], dtype=np.float64)
        self.bias_ = 0.0
        regularization = 1.0 / (self.C * len(y))
        rng = np.random.default_rng(self.random_state)
        step = 0

        for epoch in range(self.epochs):
            order = rng.permutation(len(y))
            for start in range(0, len(y), self.batch_size):
                idx = order[start:start + self.batch_size]
                Xb = X[idx]
                yb = signed_y[idx]
                sw = sample_weight[idx]

                margins = yb * (Xb @ self.weights_ + self.bias_)
                violating = margins < 1.0
                eta = self.learning_rate / (1.0 + self.decay * step)

                grad_w = regularization * self.weights_
                grad_b = 0.0

                if np.any(violating):
                    coeff = sw[violating] * yb[violating]
                    grad_w -= np.asarray(Xb[violating].T @ coeff).ravel() / len(idx)
                    grad_b = -coeff.sum() / len(idx)

                self.weights_ -= eta * grad_w
                self.bias_ -= eta * grad_b
                step += 1

            if verbose:
                print(f"epoch={epoch+1:02d} hinge_loss={self.hinge_loss(X, y):.6f}")
        return self

    def decision_function(self, X):
        return np.asarray(X @ self.weights_ + self.bias_).ravel()

    def predict(self, X):
        return (self.decision_function(X) >= 0.0).astype(np.int8)

    def hinge_loss(self, X, y):
        signed_y = np.where(np.asarray(y) == 1, 1.0, -1.0)
        return np.maximum(0.0, 1.0 - signed_y * self.decision_function(X)).mean()

In [27]:
y_train = train.loc[train_mask, LABEL_COLUMN].to_numpy(np.int8)
y_validation = train.loc[validation_mask, LABEL_COLUMN].to_numpy(np.int8)

vectorizer = ScratchTfidfVectorizer(min_df=2, max_features=100_000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train.loc[train_mask, TEXT_COLUMN].fillna(""))
X_validation = vectorizer.transform(train.loc[validation_mask, TEXT_COLUMN].fillna(""))

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("TF-IDF features:", X_train.shape[1])

X_train: (16000, 100000)
X_validation: (4000, 100000)
TF-IDF features: 100000


In [28]:
baseline = ScratchLinearSVM(
    C=1.0,
    epochs=20,
    batch_size=256,
    learning_rate=0.5,
    decay=0.02,
    class_weight=None,
    random_state=SEED,
)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_validation)
baseline_f1 = macro_f1(y_validation, baseline_pred)
print(f"Baseline validation Macro F1 = {baseline_f1:.6f}")

Baseline validation Macro F1 = 0.384710


In [29]:
C_VALUES = [0.01, 0.1, 0.5, 1, 2, 5, 10]
c_results = []

for C in C_VALUES:
    model = ScratchLinearSVM(
        C=C,
        epochs=20,
        batch_size=256,
        learning_rate=0.5,
        decay=0.02,
        class_weight=None,
        random_state=SEED,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_validation)
    score = macro_f1(y_validation, pred)
    c_results.append({"C": C, "class_weight": "None", "Macro_F1": score})
    print(f"C={C:<5} Macro F1={score:.6f}")

c_results_df = pd.DataFrame(c_results).sort_values("Macro_F1", ascending=False).reset_index(drop=True)
display(c_results_df)

C=0.01  Macro F1=0.384710
C=0.1   Macro F1=0.384710
C=0.5   Macro F1=0.384710
C=1     Macro F1=0.384710
C=2     Macro F1=0.384710
C=5     Macro F1=0.384710
C=10    Macro F1=0.384710


,C,class_weight,Macro_F1
0,0.01,None,0.38471
1,0.10,None,0.38471
2,0.50,None,0.38471
3,1.00,None,0.38471
4,2.00,None,0.38471
5,5.00,None,0.38471
6,10.00,None,0.38471


In [30]:
TOP_3_C = c_results_df.head(3)["C"].tolist()
print("Best three C values:", TOP_3_C)

weight_results = []
for C in TOP_3_C:
    for class_weight in [None, "balanced"]:
        model = ScratchLinearSVM(
            C=C,
            epochs=20,
            batch_size=256,
            learning_rate=0.5,
            decay=0.02,
            class_weight=class_weight,
            random_state=SEED,
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_validation)
        score = macro_f1(y_validation, pred)
        weight_results.append({
            "C": C,
            "class_weight": "None" if class_weight is None else "balanced",
            "Macro_F1": score,
        })
        print(f"C={C:<5} class_weight={str(class_weight):<8} Macro F1={score:.6f}")

weight_results_df = pd.DataFrame(weight_results).sort_values("Macro_F1", ascending=False).reset_index(drop=True)
display(weight_results_df)

Best three C values: [0.01, 0.1, 0.5]
C=0.01  class_weight=None     Macro F1=0.384710
C=0.01  class_weight=balanced Macro F1=0.649955
C=0.1   class_weight=None     Macro F1=0.384710
C=0.1   class_weight=balanced Macro F1=0.657864
C=0.5   class_weight=None     Macro F1=0.384710
C=0.5   class_weight=balanced Macro F1=0.658596


,C,class_weight,Macro_F1
0,0.50,balanced,0.658596
1,0.10,balanced,0.657864
2,0.01,balanced,0.649955
3,0.01,None,0.384710
4,0.10,None,0.384710
5,0.50,None,0.384710


In [31]:
best = weight_results_df.iloc[0]
BEST_C = float(best["C"])
BEST_CLASS_WEIGHT = None if best["class_weight"] == "None" else "balanced"
BEST_VALIDATION_F1 = float(best["Macro_F1"])

print("Best C:", BEST_C)
print("Best class_weight:", BEST_CLASS_WEIGHT)
print(f"Best validation Macro F1: {BEST_VALIDATION_F1:.6f}")

display(pd.DataFrame([{
    "C": BEST_C,
    "class_weight": "None" if BEST_CLASS_WEIGHT is None else BEST_CLASS_WEIGHT,
    "Macro_F1": BEST_VALIDATION_F1,
}]))

Best C: 0.5
Best class_weight: balanced
Best validation Macro F1: 0.658596


,C,class_weight,Macro_F1
0,0.5,balanced,0.658596


In [32]:
final_vectorizer = ScratchTfidfVectorizer(min_df=2, max_features=100_000, ngram_range=(1, 2))
X_full = final_vectorizer.fit_transform(train[TEXT_COLUMN].fillna(""))
X_test = final_vectorizer.transform(test[TEXT_COLUMN].fillna(""))

final_model = ScratchLinearSVM(
    C=BEST_C,
    epochs=20,
    batch_size=256,
    learning_rate=0.5,
    decay=0.02,
    class_weight=BEST_CLASS_WEIGHT,
    random_state=SEED,
)
final_model.fit(X_full, train[LABEL_COLUMN].to_numpy(np.int8), verbose=True)
print("Final model trained on all labelled rows.")

epoch=01 hinge_loss=1.000888
epoch=02 hinge_loss=1.000085
epoch=03 hinge_loss=0.995895
epoch=04 hinge_loss=0.993962
epoch=05 hinge_loss=0.991513
epoch=06 hinge_loss=0.988856
epoch=07 hinge_loss=0.987781
epoch=08 hinge_loss=0.987613
epoch=09 hinge_loss=0.984558
epoch=10 hinge_loss=0.982360
epoch=11 hinge_loss=0.980933
epoch=12 hinge_loss=0.980872
epoch=13 hinge_loss=0.980824
epoch=14 hinge_loss=0.978272
epoch=15 hinge_loss=0.977660
epoch=16 hinge_loss=0.977432
epoch=17 hinge_loss=0.977351
epoch=18 hinge_loss=0.977046
epoch=19 hinge_loss=0.975599
epoch=20 hinge_loss=0.974852
Final model trained on all labelled rows.


In [ ]:
test_predictions = final_model.predict(X_test)
submission = create_submission(test[ID_COLUMN], test_predictions, OUTPUT_PATH)

print("Saved:", OUTPUT_PATH)
print("Prediction counts:", submission["label"].value_counts().sort_index().to_dict())
display(submission.head())